# Perron Phase 3: Gemma 4 E2B LoRA Parameter-Efficient Fine-Tuning

- **Author**: Sultan Haikal (GitHub: `@yvliet`)
- **Hardware**: Kaggle Dual Tesla T4 (Single-GPU FP16 on `cuda:0`)
- **Objective**: Parameter-efficient fine-tuning of compact Gemma 4 E2B under native FP16 to resolve the Topological Disambiguation Deficit and beat the Break-Even Discrimination Bound (AUC >= 0.8298).


In [ ]:
# Step 1: Install PEFT and verify hardware environment
!pip install -q peft transformers accelerate

import os, sys, json, math, time, gzip, base64
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

# Attempt to load HF_TOKEN from Kaggle Secrets if configured
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[*] Authenticated via Kaggle Secrets HF_TOKEN.')
except Exception:
    pass

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'[*] Active Device: {device}')
if torch.cuda.is_available():
    print(f'[*] GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'[*] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# Step 2: Unpack contrastive training & validation datasets
TRAIN_B64 = '''...'''
VAL_B64 = '''...'''

train_raw = gzip.decompress(base64.b64decode(TRAIN_B64)).decode('utf-8')
val_raw = gzip.decompress(base64.b64decode(VAL_B64)).decode('utf-8')

train_records = [json.loads(l) for l in train_raw.splitlines() if l.strip()]
val_records = [json.loads(l) for l in val_raw.splitlines() if l.strip()]

print(f'[*] Extracted {len(train_records)} training records, {len(val_records)} validation records.')
Path('data').mkdir(exist_ok=True)
with open('data/train.jsonl', 'w', encoding='utf-8') as f:
    f.write(train_raw)
with open('data/val.jsonl', 'w', encoding='utf-8') as f:
    f.write(val_raw)


In [ ]:
# Step 3: Dataset class and Collate function with prompt masking
class CandidateChoiceDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=2048):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        prompt = item['prompt']
        completion = item['completion']
        full_text = prompt + completion

        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        full_ids = self.tokenizer.encode(full_text, add_special_tokens=False)
        if len(full_ids) > self.max_length:
            full_ids = full_ids[:self.max_length]

        input_ids = torch.tensor(full_ids, dtype=torch.long)
        attention_mask = torch.ones_like(input_ids)
        labels = input_ids.clone()
        prompt_len = min(len(prompt_ids), len(input_ids))
        labels[:prompt_len] = -100  # Mask prompt tokens

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'target_letter': item.get('target_letter', 'A'),
        }

def collate_fn(batch):
    max_len = max(len(b['input_ids']) for b in batch)
    input_ids_list, mask_list, labels_list, targets = [], [], [], []
    for b in batch:
        cur_len = len(b['input_ids'])
        pad_len = max_len - cur_len
        padded_input = torch.cat([b['input_ids'], torch.zeros(pad_len, dtype=torch.long)])
        padded_mask = torch.cat([b['attention_mask'], torch.zeros(pad_len, dtype=torch.long)])
        padded_labels = torch.cat([b['labels'], torch.full((pad_len,), -100, dtype=torch.long)])
        input_ids_list.append(padded_input)
        mask_list.append(padded_mask)
        labels_list.append(padded_labels)
        targets.append(b['target_letter'])
    return {
        'input_ids': torch.stack(input_ids_list),
        'attention_mask': torch.stack(mask_list),
        'labels': torch.stack(labels_list),
        'target_letters': targets,
    }


In [ ]:
# Step 4: Zero-Decode Logit Sorting Evaluation Routine
def evaluate_zero_decode(model, dataloader, tokenizer, device):
    model.eval()
    correct_top1, correct_top5, total = 0, 0, 0
    letter_tokens = [chr(ord('A') + i) for i in range(25)]
    letter_ids = [tokenizer.encode(lt, add_special_tokens=False)[-1] for lt in letter_tokens]

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            for i, target_letter in enumerate(batch['target_letters']):
                seq_len = int(attention_mask[i].sum().item())
                pred_logits = logits[i, seq_len - 2, letter_ids].cpu().numpy()
                ranked = np.argsort(-pred_logits)
                gold_idx = ord(target_letter) - ord('A')
                if 0 <= gold_idx < len(letter_tokens):
                    if ranked[0] == gold_idx:
                        correct_top1 += 1
                    if gold_idx in ranked[:5]:
                        correct_top5 += 1
                    total += 1

    return {
        'val_acc1': float(correct_top1 / total) if total > 0 else 0.0,
        'val_acc5': float(correct_top5 / total) if total > 0 else 0.0,
        'total': total
    }


In [ ]:
# Step 5: Initialize Model, Tokenizer, and LoRA Adapter
model_path = 'google/gemma-4-e2b-it'
kaggle_paths = [
    Path('/kaggle/input/gemma-4/Transformers/gemma-4-e2b-it/1'),
    Path('/kaggle/input/gemma-4-e2b-it'),
    Path('/kaggle/input/gemma-4/transformers/gemma-4-e2b-it/1')
]
for p in kaggle_paths:
    if p.exists():
        model_path = str(p)
        print(f'[*] Located local Kaggle model: {model_path}')
        break

print(f'[*] Loading tokenizer from {model_path}...')
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'[*] Loading base model in native FP16 on {device}...')
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=dtype,
    trust_remote_code=True
).to(device)

if hasattr(model, 'config'):
    model.config.use_cache = False

model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=r'.*language_model.*(q_proj|v_proj|k_proj|o_proj)$',
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
# Step 6: Execute LoRA Supervised Fine-Tuning
epochs = 3
batch_size = 1
accum_steps = 8
lr = 2e-4

train_ds = CandidateChoiceDataset(train_records, tokenizer, max_length=2048)
val_ds = CandidateChoiceDataset(val_records, tokenizer, max_length=2048)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)
total_steps = math.ceil(len(train_loader) / accum_steps) * epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

output_dir = Path('perron-gemma4-e2b-lora')
output_dir.mkdir(exist_ok=True)
history = []
best_acc5 = -1.0

print(f'[*] Training started: {epochs} epochs, {total_steps} optimizer steps.')
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader, start=1):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available(), dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)
            loss = outputs.loss / accum_steps

        scaler.scale(loss).backward()
        epoch_loss += loss.item() * accum_steps

        if step % accum_steps == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

    avg_loss = epoch_loss / len(train_loader)
    val_metrics = evaluate_zero_decode(model, val_loader, tokenizer, device)
    print(f'Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f} | Val Acc@1: {val_metrics["val_acc1"]:.4f} | Val Acc@5: {val_metrics["val_acc5"]:.4f}')
    history.append({'epoch': epoch, 'loss': avg_loss, **val_metrics})

    if val_metrics['val_acc5'] > best_acc5:
        best_acc5 = val_metrics['val_acc5']
        print(f'[+] New best checkpoint: saving adapter to {output_dir}...')
        model.save_pretrained(str(output_dir))
        tokenizer.save_pretrained(str(output_dir))

with open(output_dir / 'training_telemetry.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'[DONE] Phase 3 LoRA Fine-Tuning Complete! Best Val Acc@5: {best_acc5:.4f}')


In [ ]:
# Step 7: Export final adapter zip archive for distribution
!zip -r perron-gemma4-e2b-lora.zip perron-gemma4-e2b-lora
print('[+] Adapter archive created: perron-gemma4-e2b-lora.zip')
!ls -lh perron-gemma4-e2b-lora.zip
